In [27]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("raiyanjahangir1939/ecg-arrhythmia-mitbih-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ecg-arrhythmia-mitbih-dataset' dataset.
Path to dataset files: /kaggle/input/ecg-arrhythmia-mitbih-dataset


In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, recall_score, precision_score, confusion_matrix, f1_score
import warnings
warnings.filterwarnings('ignore')


In [34]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Conv2D, MaxPooling2D, Dropout, Flatten

In [ ]:

df_train = pd.read_csv(os.path.join(path,"mitbih_train.csv"),header= None)
df_test = pd.read_csv(os.path.join(path,"mitbih_test.csv"),header= None)

# Output Variable trace
# 0: Normal beat (N)
# 1: Supraventricular ectopic beat (S)
# 2: Ventricular ectopic beat (V)
# 3: Fusion beat (F)
# 4: Unknown beat (Q)

In [ ]:
def data_explore(df1,df2):
  #shape
  print("The shape of training data is: ",df1.shape)
  print("The shape of testing data is: ",df2.shape)

  #Null counts
  print("The null counts on training data: ",df1.isnull().sum())
  print("The null counts on testing data: ",df2.isnull().sum())

  #value count of output
  print(" How many values are there in traingin target column: ",df1[187].value_counts())
  print(" How many values are there in testing target column: ",df2[187].value_counts())

  #Duplicate check
  print(" How many values are duplicate in traingin  column: ",df1.duplicated().sum())
  print(" How many values are duplicate in testing  column: ",df2.duplicated().sum())

  #Describe
  print(df1.describe().mean())
  print(df2.describe().mean())

  #Info
  print("Information of training: ",df1.info())
  print("Information of testing: ",df2.info())

In [ ]:
data_explore(df_train,df_test)

In [ ]:
# Visualization of all types of heart beat.
fig, axes = plt.subplots(5,1,figsize = (12,8))
for c in range(5):
  data = df_train[df_train[187] == c].iloc[50,:187]
  axes[c].plot(data)
  axes[c].set_title(f"Visualize of {c}")
plt.tight_layout()
plt.show()
# 0: Normal beat (N)
# 1: Supraventricular ectopic beat (S)
# 2: Ventricular ectopic beat (V)
# 3: Fusion beat (F)
# 4: Unknown beat (Q)

# EDA
Here every thing is in float datatype. There is 5 types of ecg's. And normal ecg's are more than any other one. So we will need to keep attention that the model should also learn properly about others. There is no duplicate or null values in the dataset.


In [ ]:
# Preparing the dataset for the model
X = df_train.drop(187, axis=1)
y = df_train[187]

In [ ]:
# Creating training and testing data from the training dataset so that we could validate the model before.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify= y)

In [ ]:
# Making the model ( Atfirst the Random Forest then we will decide about the Deep Learning models)
rf = RandomForestClassifier( n_estimators=100,random_state= 42,n_jobs= -1)
rf.fit(X_train,y_train)
y_predic = rf.predict(X_test)


In [ ]:
def validation(y_test, y_pred):
  print("The classification report : ")
  print(classification_report(y_test,y_pred))
  print("The Confusion Matrix : ")
  print(confusion_matrix(y_test,y_pred))


In [ ]:
validation(y_test,y_predic)

In [ ]:
# THe score of "1.0" & "3.0" is not ssatisfactory. And the recall value of "0.0" is showing like it is overfitting.
#  Need to apply some tecniqs to overcome it.

In [ ]:
rf_w = RandomForestClassifier( n_estimators=100,random_state= 42,n_jobs= -1,class_weight = 'balanced_subsample')
rf_w.fit(X_train,y_train)
y_predic_w = rf_w.predict(X_test)
validation(y_test,y_predic_w)

In [ ]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(X_train, y_train)
print(np.bincount(y_res))   # all classes now equal

rf_os = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf_os.fit(X_res, y_res)
pred_os = rf_os.predict(X_test)   # X_val is untouched

validation(y_test, pred_os)

In [ ]:
results = {}
for name, pred in [('baseline', y_predic), ('class_weight', y_predic_w), ('oversample', pred_os)]:
    f1s = f1_score(y_test, pred, average=None)
    results[name] = list(f1s) + [f1_score(y_test, pred, average='macro')]

table = pd.DataFrame(results, index=['F1_c0', 'F1_c1', 'F1_c2', 'F1_c3', 'F1_c4', 'macro_F1']).T
print(table.round(3))

In [ ]:
#               F1_c0  F1_c1  F1_c2  F1_c3  F1_c4  macro_F1
# baseline      0.986  0.754  0.927  0.720  0.974     0.873
# class_weight  0.986  0.748  0.933  0.764  0.977     0.882
# oversample    0.988  0.783  0.941  0.759  0.981     0.891

# Final Decision
After experimenting we have decided that we will use Oversamplimg to handle the training. As we have much data on 0( Normal beat) model was learning more on that. For taking a good prediction we have experimented several options. And I am fixing with the Oversampling method.